# FinTSB Hybrid QNN (Full Run)

This notebook trains a plain NN baseline and a hybrid QNN with linear concatenation on the real FinTSB forecasting dataset. It uses the same merge logic as `FinTSB-main/data/gendata.py` and runs on the full train/valid/test splits with no synthetic fallback and no smoke subset.

In [1]:
import copy
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
BASE_DIR = Path('/home/sammarv/quantum_corrosion')
FINTSB_DIR = BASE_DIR / 'FinTSB-main'
RAW_DIR = FINTSB_DIR / 'data' / 'FinTSB_cn'
MERGED_PATH = RAW_DIR / 'merged_dataset.pkl'
OUT_DIR = BASE_DIR / 'results' / 'fintsb_hybrid_qnn_full'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATTERNS = ['fluctuation', 'rise', 'fall', 'extreme']


def merge_fintsb_data(raw_dir: Path) -> pd.DataFrame:
    merged_data = []
    for pattern in PATTERNS:
        pattern_dir = raw_dir / pattern
        all_files = sorted(pattern_dir.glob('dataset_*.pkl'))
        if not all_files:
            raise FileNotFoundError(f'Missing FinTSB files in {pattern_dir}')
        for source_id, file_path in enumerate(all_files):
            df = pd.read_pickle(file_path)
            df = df.reset_index()
            df['source'] = source_id
            df = df.set_index(['datetime', 'instrument'])
            merged_data.append(df)

    if not merged_data:
        raise FileNotFoundError(f'No FinTSB pkl files found under {raw_dir}')

    final_df = pd.concat(merged_data)
    raw_dir.mkdir(parents=True, exist_ok=True)
    final_df.to_pickle(MERGED_PATH)
    return final_df


if MERGED_PATH.exists():
    df_raw = pd.read_pickle(MERGED_PATH)
    print('Loaded merged dataset:', MERGED_PATH)
else:
    df_raw = merge_fintsb_data(RAW_DIR)
    print('Created merged dataset:', MERGED_PATH)

print('Raw shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))

FileNotFoundError: Missing FinTSB files in /home/sammarv/quantum_corrosion/FinTSB-main/data/FinTSB_cn/fluctuation

In [ ]:
def normalize_index(df: pd.DataFrame) -> pd.DataFrame:
    dfn = df.copy()
    if not isinstance(dfn.index, pd.MultiIndex):
        if {'datetime', 'instrument'}.issubset(dfn.columns):
            dfn['datetime'] = pd.to_datetime(dfn['datetime'])
            dfn = dfn.set_index(['datetime', 'instrument'])
        else:
            raise ValueError('Expected a MultiIndex or datetime/instrument columns')

    if list(dfn.index.names) != ['datetime', 'instrument']:
        dfn = dfn.swaplevel().sort_index()
        dfn.index = dfn.index.set_names(['datetime', 'instrument'])
    else:
        dfn = dfn.sort_index()

    return dfn


df = normalize_index(df_raw)
if 'source' not in df.columns:
    df['source'] = 0
if 'label' not in df.columns:
    raise ValueError('FinTSB dataframe must include a label column')

feature_cols = [c for c in df.columns if c not in ['label', 'source']]
print('Feature count:', len(feature_cols))
print('Rows:', len(df))
print('Unique dates:', df.index.get_level_values('datetime').nunique())
print('Unique instruments:', df.index.get_level_values('instrument').nunique())

In [ ]:
SEQ_LEN = 20


def build_sequences(df_in: pd.DataFrame, seq_len: int = 20):
    dfr = df_in.reset_index()
    dfr['datetime'] = pd.to_datetime(dfr['datetime'])
    dfr = dfr.sort_values(['source', 'instrument', 'datetime'])

    feat_cols_local = [c for c in dfr.columns if c not in ['datetime', 'instrument', 'source', 'label']]
    feat_dim = len(feat_cols_local)
    feats = []
    labels = []
    dates = []

    for (_, _), g in dfr.groupby(['source', 'instrument'], sort=False):
        Xg = g[feat_cols_local].to_numpy(dtype=np.float32)
        yg = g['label'].to_numpy(dtype=np.float32)
        dg = pd.to_datetime(g['datetime']).to_numpy()

        for i in range(len(g)):
            start = max(0, i - seq_len + 1)
            win = Xg[start:i + 1]
            if len(win) < seq_len:
                pad = np.zeros((seq_len - len(win), feat_dim), dtype=np.float32)
                win = np.concatenate([pad, win], axis=0)
            feats.append(win)
            labels.append(yg[i])
            dates.append(dg[i])

    X = np.stack(feats).astype(np.float32)
    y = np.array(labels, dtype=np.float32)
    dates = pd.to_datetime(np.array(dates))
    return X, y, dates


X_all, y_all, date_all = build_sequences(df, seq_len=SEQ_LEN)
print('Sequence tensor:', X_all.shape)
print('Label tensor:', y_all.shape)

In [ ]:
TRAIN_START, TRAIN_END = pd.Timestamp('2000-01-03'), pd.Timestamp('2000-09-01')
VALID_START, VALID_END = pd.Timestamp('2000-09-04'), pd.Timestamp('2000-10-06')
TEST_START, TEST_END = pd.Timestamp('2000-10-09'), pd.Timestamp('2000-12-15')

tr_mask = (date_all >= TRAIN_START) & (date_all <= TRAIN_END)
va_mask = (date_all >= VALID_START) & (date_all <= VALID_END)
te_mask = (date_all >= TEST_START) & (date_all <= TEST_END)

X_tr, y_tr, d_tr = X_all[tr_mask], y_all[tr_mask], date_all[tr_mask]
X_va, y_va, d_va = X_all[va_mask], y_all[va_mask], date_all[va_mask]
X_te, y_te, d_te = X_all[te_mask], y_all[te_mask], date_all[te_mask]

print('Split sizes -> train:', len(y_tr), 'valid:', len(y_va), 'test:', len(y_te))
if min(len(y_tr), len(y_va), len(y_te)) == 0:
    raise ValueError('One or more splits are empty. Check dataset dates and split bounds.')

In [ ]:
n_feat = X_tr.shape[-1]
scaler = StandardScaler()
scaler.fit(X_tr.reshape(-1, n_feat))


def apply_scaler(X):
    shape = X.shape
    return scaler.transform(X.reshape(-1, n_feat)).reshape(shape).astype(np.float32)


X_tr = apply_scaler(X_tr)
X_va = apply_scaler(X_va)
X_te = apply_scaler(X_te)


def date_to_int64(d):
    return pd.to_datetime(d).astype('datetime64[ns]').astype(np.int64)


tr_ds = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32),
    torch.tensor(date_to_int64(d_tr), dtype=torch.int64),
)
va_ds = TensorDataset(
    torch.tensor(X_va, dtype=torch.float32),
    torch.tensor(y_va, dtype=torch.float32),
    torch.tensor(date_to_int64(d_va), dtype=torch.int64),
)
te_ds = TensorDataset(
    torch.tensor(X_te, dtype=torch.float32),
    torch.tensor(y_te, dtype=torch.float32),
    torch.tensor(date_to_int64(d_te), dtype=torch.int64),
)

BATCH_SIZE = 128
train_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False)

print('DataLoaders ready.')

In [ ]:
n_qubits = 4
n_layers = 2


def make_quantum_device(n_q):
    try:
        dev = qml.device('lightning.gpu', wires=n_q)
        return dev, 'adjoint'
    except Exception:
        dev = qml.device('default.qubit', wires=n_q)
        return dev, 'backprop'


qdev, diff_method = make_quantum_device(n_qubits)


@qml.qnode(qdev, interface='torch', diff_method=diff_method)
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class BatchedQuantumLayer(nn.Module):
    def __init__(self, n_l, n_q, q_node):
        super().__init__()
        self.n_qubits = n_q
        self.qnode = q_node
        self.weights = nn.Parameter(torch.empty(n_l, n_q).uniform_(-np.pi, np.pi))

    def forward(self, x):
        out = self.qnode(x, self.weights)
        if isinstance(out, list):
            out = torch.stack([
                o if isinstance(o, torch.Tensor) else torch.as_tensor(o, dtype=x.dtype, device=x.device)
                for o in out
            ])
        if not isinstance(out, torch.Tensor):
            out = torch.as_tensor(out, dtype=x.dtype, device=x.device)
        out = out.float().view(self.n_qubits, x.shape[0]).t().contiguous()
        return out


class PlainNN(nn.Module):
    def __init__(self, seq_len, n_feat):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(seq_len * n_feat, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class HybridQNNLinearConcat(nn.Module):
    def __init__(self, seq_len, n_feat):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(seq_len * n_feat, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.GELU(),
        )
        self.qnn_proj = nn.Sequential(nn.Linear(128, n_qubits), nn.Sigmoid())
        self.qnn = BatchedQuantumLayer(n_layers, n_qubits, qnode)
        self.classical_skip = nn.Sequential(nn.Linear(128, 64), nn.ReLU())
        self.fusion = nn.Linear(64 + n_qubits, 1)

    def forward(self, x):
        feats = self.encoder(x)
        q_in = self.qnn_proj(feats) * (2.0 * np.pi)
        q_feats = self.qnn(q_in)
        c_feats = self.classical_skip(feats)
        fused = torch.cat([c_feats, q_feats], dim=1)
        return self.fusion(fused).squeeze(1)


baseline_model = PlainNN(seq_len=SEQ_LEN, n_feat=n_feat).to(device)
hybrid_model = HybridQNNLinearConcat(seq_len=SEQ_LEN, n_feat=n_feat).to(device)

print('Baseline params:', sum(p.numel() for p in baseline_model.parameters() if p.requires_grad))
print('Hybrid params:', sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad))

In [ ]:
def rank_mse_loss(pred, target, rank_weight=0.05, mse_weight=1.0):
    mse = F.mse_loss(pred, target)
    pred_diff = pred.unsqueeze(1) - pred.unsqueeze(0)
    target_diff = target.unsqueeze(1) - target.unsqueeze(0)
    rank = F.relu(-(pred_diff * target_diff)).mean()
    return mse_weight * mse + rank_weight * rank


def pearson_np(a, b):
    if len(a) < 2:
        return np.nan
    sa = np.std(a)
    sb = np.std(b)
    if sa < 1e-12 or sb < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def rankdata_avg(x):
    x = np.asarray(x)
    order = np.argsort(x)
    ranks = np.empty(len(x), dtype=np.float64)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        avg_rank = 0.5 * (i + j) + 1.0
        ranks[order[i:j + 1]] = avg_rank
        i = j + 1
    return ranks


def spearman_np(a, b):
    if len(a) < 2:
        return np.nan
    return pearson_np(rankdata_avg(a), rankdata_avg(b))


def evaluate_regression(model, loader):
    model.eval()
    all_pred = []
    all_y = []
    all_d = []

    with torch.no_grad():
        for xb, yb, db in loader:
            xb = xb.to(device)
            pb = model(xb)
            all_pred.append(pb.cpu().numpy())
            all_y.append(yb.numpy())
            all_d.append(db.numpy())

    pred = np.concatenate(all_pred)
    y = np.concatenate(all_y)
    d = np.concatenate(all_d)

    mse = float(np.mean((pred - y) ** 2))
    mae = float(np.mean(np.abs(pred - y)))

    day_ic = []
    day_rankic = []
    for day in np.unique(d):
        idx = np.where(d == day)[0]
        if len(idx) < 2:
            continue
        ic = pearson_np(pred[idx], y[idx])
        ric = spearman_np(pred[idx], y[idx])
        if not np.isnan(ic):
            day_ic.append(ic)
        if not np.isnan(ric):
            day_rankic.append(ric)

    out = {
        'MSE': mse,
        'MAE': mae,
        'IC': float(np.mean(day_ic)) if day_ic else float('nan'),
        'RankIC': float(np.mean(day_rankic)) if day_rankic else float('nan'),
        'n_samples': int(len(y)),
    }
    return out, pred, y, d


def train_model(model, train_loader, valid_loader, epochs=3, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_valid_ic = -1e9
    best_state = None
    history = []

    for ep in range(epochs):
        model.train()
        run_loss = 0.0
        n_seen = 0

        for xb, yb, _ in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = rank_mse_loss(pred, yb)
            loss.backward()
            optimizer.step()

            run_loss += loss.item() * xb.size(0)
            n_seen += xb.size(0)

        train_loss = run_loss / max(n_seen, 1)
        valid_metrics, _, _, _ = evaluate_regression(model, valid_loader)
        history.append({'epoch': ep + 1, 'train_loss': train_loss, **valid_metrics})
        print(f'Epoch {ep + 1}/{epochs} train_loss={train_loss:.6f} valid={valid_metrics}')

        current_ic = valid_metrics['IC'] if not math.isnan(valid_metrics['IC']) else -1e9
        if current_ic > best_valid_ic:
            best_valid_ic = current_ic
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


EPOCHS = 3
baseline_model, baseline_history = train_model(baseline_model, train_loader, valid_loader, epochs=EPOCHS, lr=1e-3)
hybrid_model, hybrid_history = train_model(hybrid_model, train_loader, valid_loader, epochs=EPOCHS, lr=1e-3)

baseline_metrics, baseline_pred, baseline_y, baseline_d = evaluate_regression(baseline_model, test_loader)
hybrid_metrics, hybrid_pred, hybrid_y, hybrid_d = evaluate_regression(hybrid_model, test_loader)

print('Baseline test metrics:', baseline_metrics)
print('Hybrid QNN test metrics:', hybrid_metrics)

In [ ]:
baseline_weights_path = OUT_DIR / 'baseline_nn_best.pt'
hybrid_weights_path = OUT_DIR / 'hybrid_qnn_linear_concat_best.pt'
metrics_path = OUT_DIR / 'metrics_full.json'
baseline_preds_path = OUT_DIR / 'baseline_nn_pred_full.csv'
hybrid_preds_path = OUT_DIR / 'hybrid_qnn_pred_full.csv'


torch.save(baseline_model.state_dict(), str(baseline_weights_path))
torch.save(hybrid_model.state_dict(), str(hybrid_weights_path))

baseline_pred_df = pd.DataFrame({
    'datetime_ns': baseline_d,
    'score': baseline_pred,
    'label': baseline_y,
    'model': 'baseline_nn',
})
baseline_pred_df['datetime'] = pd.to_datetime(baseline_pred_df['datetime_ns'])
baseline_pred_df = baseline_pred_df[['datetime', 'score', 'label', 'model']]
baseline_pred_df.to_csv(baseline_preds_path, index=False)

hybrid_pred_df = pd.DataFrame({
    'datetime_ns': hybrid_d,
    'score': hybrid_pred,
    'label': hybrid_y,
    'model': 'hybrid_qnn_linear_concat',
})
hybrid_pred_df['datetime'] = pd.to_datetime(hybrid_pred_df['datetime_ns'])
hybrid_pred_df = hybrid_pred_df[['datetime', 'score', 'label', 'model']]
hybrid_pred_df.to_csv(hybrid_preds_path, index=False)

metrics_payload = {
    'seed': SEED,
    'epochs': EPOCHS,
    'seq_len': SEQ_LEN,
    'batch_size': BATCH_SIZE,
    'n_qubits': n_qubits,
    'n_layers': n_layers,
    'train_samples': len(tr_ds),
    'valid_samples': len(va_ds),
    'test_samples': len(te_ds),
    'baseline_metrics': baseline_metrics,
    'hybrid_metrics': hybrid_metrics,
}
with open(metrics_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)

print('Saved baseline weights:', baseline_weights_path)
print('Saved hybrid weights:', hybrid_weights_path)
print('Saved metrics:', metrics_path)
print('Saved baseline predictions:', baseline_preds_path)
print('Saved hybrid predictions:', hybrid_preds_path)